# arange-fancy-index-cross-entropy — ex1: pick per-sample target logits via logits[arange(B), target]

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `arange-fancy-index-cross-entropy`. Running the final beacon cell reports progress against the `Loss: arange fancy-index cross-entropy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Loss: arange fancy-index cross-entropy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`arange-fancy-index-cross-entropy`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "arange-fancy-index-cross-entropy"
DD_SUBTOPIC = "Loss: arange fancy-index cross-entropy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `logits[arange(B), labels]` fancy-index — quick refresher

Per-sample target-logit extraction shows up in EVERY classification loss. You have `logits` of shape `(B, C)` and `labels` of shape `(B,)`. You want a `(B,)` vector where position `i` holds `logits[i, labels[i]]`.

**The wrong way (slow, breaks autograd).** Python loop:
```python
picked = t.stack([logits[i, labels[i]] for i in range(B)])
```

**The right way (vectorized).** Use advanced indexing with `arange`:
```python
picked = logits[t.arange(B), labels]    # shape (B,)
```

Mechanics: when you index with TWO 1-D tensors of the same length, PyTorch pairs them positionally. `arange(B) = [0,1,2,...,B-1]` and `labels = [l0, l1, ...]` zip to `[(0,l0), (1,l1), ...]`, picking one element per row.

**Why `arange`, not `slice(None)`.** A plain `logits[:, labels]` would broadcast — `(B, B)` output — not `(B,)`. The `arange` makes the row axis advance in lockstep with the column axis.

### Exercise 1 — pick per-sample target logits via logits[arange(B), target]

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply NumPy/PyTorch advanced indexing with `arange(B)` paired with a `(B,)` index tensor to extract one column per row, producing a `(B,)` vector of per-sample target logits.
> Keywords: arange, fancy-indexing, advanced-indexing, per-sample, target-logit
> ```

**KCs targeted:** `arange-fancy-index-cross-entropy`, `logsumexp-cross-entropy`

Implement `pick_target_logits(logits, target)`. Returns a `(B,)` tensor where position `i` is `logits[i, target[i]]`.

Inputs:
- `logits`: shape `(B, C)`, float.
- `target`: shape `(B,)`, integer class indices in `[0, C)`.

Output: shape `(B,)`, same dtype as `logits`.

**Vectorized one-liner:**
```python
logits[t.arange(B), target]
```

Mechanics: when you index a 2-D tensor with TWO 1-D tensors of the same length, PyTorch pairs them positionally. `arange(B) = [0, 1, ..., B-1]` and `target = [t0, t1, ...]` zip into `[(0, t0), (1, t1), ...]`. One element per row.

**Forbidden:** Python for-loops. The drill is specifically the vectorized pattern. A loop over `range(B)` would be `O(B)` Python overhead and would break grad accumulation (it materializes one scalar at a time, with separate Recipes).

**Why `arange`, not `:`.** `logits[:, target]` broadcasts to `(B, B)` — every row indexed by every target — not what we want. The `arange` advances the row axis in lockstep with the column axis.

In [ ]:
def pick_target_logits(logits: Tensor, target: Tensor) -> Tensor:
    """Return logits[arange(B), target] — per-sample target logits, shape (B,)."""
    raise NotImplementedError()


def _test_ex1():
    # --- the trivial case: B=3, target picks one column per row ---
    logits = t.tensor([
        [10.0, 20.0, 30.0],
        [40.0, 50.0, 60.0],
        [70.0, 80.0, 90.0],
    ])
    target = t.tensor([0, 1, 2])
    got = pick_target_logits(logits, target)
    assert got.shape == (3,), f'shape: {got.shape}'
    assert t.allclose(got, t.tensor([10.0, 50.0, 90.0])), f'value: {got}'

    # --- target on the same column for every row → constant slice ---
    target_const = t.tensor([1, 1, 1])
    got_const = pick_target_logits(logits, target_const)
    assert t.allclose(got_const, t.tensor([20.0, 50.0, 80.0])), (
        f'target=1 for all rows must pick column 1: {got_const}'
    )

    # --- DO NOT broadcast — output is (B,), NOT (B, B) ---
    assert got_const.shape == (3,), (
        f'output must be (3,) not (3,3); did you write logits[:, target]? '
        f'Got {got_const.shape}'
    )
    # Explicit comparison against the wrong (broadcasting) result.
    broadcast_wrong = logits[:, target_const]
    assert broadcast_wrong.shape == (3, 3), 'sanity: logits[:, target] does broadcast'
    assert got_const.shape != broadcast_wrong.shape, (
        'your result must NOT match the broadcasting version'
    )

    # --- larger batch: random logits, target picks one per row ---
    rng = t.Generator().manual_seed(0)
    big_logits = t.randn(32, 10, generator=rng)
    big_target = t.randint(0, 10, (32,), generator=rng)
    got_big = pick_target_logits(big_logits, big_target)
    assert got_big.shape == (32,)
    # Witness via a Python loop (slow but obviously correct).
    expected_big = t.tensor([big_logits[i, big_target[i].item()].item() for i in range(32)])
    assert t.allclose(got_big, expected_big), 'value mismatch on (32, 10) batch'

    # --- dtype preserved ---
    logits_d = t.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=t.float64)
    target_d = t.tensor([0, 1])
    out_d = pick_target_logits(logits_d, target_d)
    assert out_d.dtype == t.float64, f'dtype must be preserved, got {out_d.dtype}'

    # --- composes with cross-entropy: lse - picked is per-sample CE loss ---
    lse = t.logsumexp(logits, dim=-1)
    picked = pick_target_logits(logits, target)
    per_sample_ce = lse - picked
    assert per_sample_ce.shape == (3,), 'pick_target_logits must compose into CE without reshape'
    # Sanity: row 0 target 0, logits [10,20,30] → lse ~= 30+log(1+e^-10+e^-20) ~ 30, picked=10 → ~20
    assert per_sample_ce[0].item() > 19.0 and per_sample_ce[0].item() < 21.0, (
        f'CE row 0 sanity: {per_sample_ce[0].item()}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def pick_target_logits(logits: Tensor, target: Tensor) -> Tensor:
    B = logits.shape[0]
    # advanced indexing: arange row axis paired with target column axis
    return logits[t.arange(B), target]
```

**Two 1-D index tensors = positional pairing.** This is the key rule of NumPy / PyTorch advanced indexing: when you index with multiple 1-D integer tensors of the same length, they get zipped into coordinate tuples. `logits[[0,1,2], [t0,t1,t2]]` is `[logits[0,t0], logits[1,t1], logits[2,t2]]`. The `arange(B)` is just a compact way to spell `[0, 1, ..., B-1]`.

**Why this matters for autograd.** Advanced indexing is differentiable: the gradient w.r.t. `logits` is a sparse tensor that scatters the upstream `(B,)` gradient back to the original `(B, C)` positions (zeros everywhere except `(i, target[i])`). A Python for-loop would materialize each scalar separately, breaking this clean gradient path.

**Common alternative — `gather`.** `logits.gather(dim=1, index=target.unsqueeze(-1)).squeeze(-1)` does the same thing. Slightly more verbose but generalizes cleanly to higher-rank tensors. For the `(B, C)` classification case, `arange` indexing is the idiomatic move.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()